# Fly (*Drosophila*) LDSC Cell-Type-Specific Heritability Analysis

End-to-end pipeline: DGRP2 genotypes → GWAS → LDSC h2-cts enrichment.

| | |
|---|---|
| **Genome** | dm6 |
| **Reference panel** | DGRP2 (205 inbred lines, WGS-based) |
| **Phenotype** | Mean longevity, both sexes averaged (DGRPool Study 1, Arya et al. 2010) |
| **Method** | LDSC partitioned heritability, cell-type-specific (h2-cts) |

Run cells top-to-bottom. Each section prints a status summary when done.

##### 0. Imports & Configuration

In [ ]:
import os, re, subprocess, glob, json, gzip, urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
%matplotlib inline

from pathlib import Path

# ── Set this to the root of your fly-ldsc repo ──────────────────────────────
BASE_DIR    = Path("/mnt/hdd_1/rediet/fly-ldsc")

FLY_CHROMS  = ["2L", "2R", "3L", "3R", "4", "X"]
DGRP_PREFIX = str(BASE_DIR / "data" / "reference" / "DGRP")
TOOLS_DIR   = BASE_DIR / "tools"
LDSC_DIR    = TOOLS_DIR / "ldsc"

print(f"BASE_DIR : {BASE_DIR}")
print(f"Exists   : {BASE_DIR.exists()}")

##### 1. Setup LDSC

Clone the LDSC repository and configure a Python 2.7 conda environment with all
required dependencies (`numpy`, `scipy`, `pandas`, `bitarray`, `pybedtools`).

In [ ]:
TOOLS_DIR.mkdir(exist_ok=True)

# Clone LDSC if needed
if not LDSC_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/bulik/ldsc.git", str(LDSC_DIR)], check=True)

# Locate Python 2.7 in ldsc27 conda env
_conda = subprocess.run(["conda", "env", "list", "--json"], capture_output=True, text=True, check=True)
_envs  = json.loads(_conda.stdout)["envs"]
_match = [e for e in _envs if "ldsc27" in e]
if not _match:
    raise RuntimeError("conda env 'ldsc27' not found — run: conda create -n ldsc27 python=2.7")
python27_path = str(Path(_match[0]) / "bin" / "python")

subprocess.run(["chmod", "+x", str(LDSC_DIR / "ldsc.py")], check=True)
subprocess.run(["chmod", "+x", str(LDSC_DIR / "make_annot.py")], check=True)

print(f"LDSC dir        : {LDSC_DIR}")
print(f"Python 2.7 path : {python27_path}")

##### 2. Validate DGRP Reference Files

Check that plink binary files (`.bed/.bim/.fam`) exist for all 6 chromosome arms.
Each file was generated from the DGRP2 whole-genome sequencing VCF (dm6 assembly).

In [ ]:
missing = []
for ch in FLY_CHROMS:
    for ext in [".bed", ".bim", ".fam"]:
        f = f"{DGRP_PREFIX}.{ch}{ext}"
        if not os.path.exists(f):
            missing.append(f)

if missing:
    print("MISSING files:")
    for f in missing: print(f"  {f}")
else:
    print("All DGRP2 reference files present:")
    for ch in FLY_CHROMS:
        n = sum(1 for _ in open(f"{DGRP_PREFIX}.{ch}.bim"))
        print(f"  chr{ch}: {n:,} SNPs")

##### 3. Discover Cell-Type BED Files

Each BED file defines open-chromatin peaks for one cell type (dm6 coordinates).
Files live in `data/peaks/` — one `.bed` per cell type.

In [ ]:
peaks_dir = BASE_DIR / "data" / "peaks"
peaks_dir.mkdir(parents=True, exist_ok=True)

def sanitize(name):
    return re.sub(r"[^A-Za-z0-9_\-]", "_", name)

cell_type_beds = {}
for f in sorted(os.listdir(peaks_dir)):
    if f.endswith(".bed") and not f.startswith("."):
        safe = sanitize(os.path.splitext(f)[0])
        cell_type_beds[safe] = str(peaks_dir / f)

all_cell_types = sorted(cell_type_beds.keys())

if not all_cell_types:
    print("No BED files found in data/peaks/")
else:
    print(f"{len(all_cell_types)} cell types found:")
    for ct in all_cell_types[:10]:
        n = sum(1 for _ in open(cell_type_beds[ct]))
        print(f"  {ct}: {n:,} peaks")
    if len(all_cell_types) > 10:
        print(f"  ... and {len(all_cell_types)-10} more")

##### 4. Generate Cell-Type Annotations

Run `ldsc make_annot.py` to convert each BED file into per-chromosome `.annot.gz` files.
Each SNP in the DGRP2 reference panel is marked 1 if it falls in a peak, 0 otherwise.

In [ ]:
annot_dir = BASE_DIR / "data" / "annotations"
annot_dir.mkdir(parents=True, exist_ok=True)

ldsc27_bin = str(Path(python27_path).parent)
env = os.environ.copy()
env["PATH"] = ldsc27_bin + ":" + env.get("PATH", "")

for ct in all_cell_types:
    if all(os.path.exists(f"{annot_dir}/{ct}.{ch}.annot.gz") for ch in FLY_CHROMS):
        print(f"  {ct}: exists, skipping")
        continue
    print(f"  {ct}...", end=" ", flush=True)
    for ch in FLY_CHROMS:
        out = f"{annot_dir}/{ct}.{ch}.annot.gz"
        if os.path.exists(out):
            continue
        r = subprocess.run([
            python27_path, str(LDSC_DIR / "make_annot.py"),
            "--bed-file",   cell_type_beds[ct],
            "--bimfile",    f"{DGRP_PREFIX}.{ch}.bim",
            "--annot-file", out,
        ], capture_output=True, text=True, env=env)
        if r.returncode != 0:
            print(f"ERROR chr{ch}: {r.stderr[:100]}")
    print("done")
print("\nAll annotations generated")

##### 5. Calculate LD Scores

Compute per-cell-type, per-chromosome LD scores using `ldsc.py --l2`.
Uses a 1 Mb LD window, matching the DGRP population size (~200 lines).

In [ ]:
import concurrent.futures, multiprocessing

ldscore_dir = BASE_DIR / "data" / "ldscores"
ldscore_dir.mkdir(parents=True, exist_ok=True)

def calc_ld(args):
    ct, ch = args
    d   = ldscore_dir / ct
    d.mkdir(exist_ok=True)
    out = d / f"{ct}.{ch}.l2.ldscore.gz"
    if out.exists():
        return f"  [{ct}] chr{ch}: exists"
    try:
        subprocess.run([
            python27_path, str(LDSC_DIR / "ldsc.py"),
            "--l2", "--bfile", f"{DGRP_PREFIX}.{ch}",
            "--ld-wind-kb", "1000",
            "--annot",      str(annot_dir / f"{ct}.{ch}.annot.gz"),
            "--thin-annot", "--out", str(d / f"{ct}.{ch}"),
        ], check=True, capture_output=True)
        return f"  [{ct}] chr{ch}: done"
    except subprocess.CalledProcessError as e:
        return f"  ERROR [{ct}] chr{ch}"

tasks = [(ct, ch) for ct in all_cell_types for ch in FLY_CHROMS]
workers = min(multiprocessing.cpu_count() - 1, 8)
print(f"Running {len(tasks)} tasks with {workers} workers...")

with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
    for res in ex.map(calc_ld, tasks):
        print(res)
print("\nAll LD scores calculated")

##### 6. Phenotype Preparation

Download DGRPool Study 1 longevity data (Arya et al. 2010) and average male + female
measurements per DGRP line. Reformat sample IDs to match the DGRP2 `.fam` file.

In [ ]:
gwas_dir  = BASE_DIR / "data" / "gwas"
gwas_dir.mkdir(parents=True, exist_ok=True)

RAW_FILE   = gwas_dir / "study1_longevity_raw.tsv"
PHENO_FILE = gwas_dir / "lifespan_both_sex.pheno"
PHENO_NAME = "longevity_both_sex"
SUMMARY_URL = "https://dgrpool.epfl.ch/studies/1/get_file?name=summary.tsv"

if not RAW_FILE.exists():
    print("Downloading DGRPool Study 1...")
    urllib.request.urlretrieve(SUMMARY_URL, RAW_FILE)
else:
    print(f"Already downloaded: {RAW_FILE.name}")

with gzip.open(RAW_FILE, "rt") as f:
    df_raw = pd.read_csv(f, sep="\t")

print(f"Columns : {list(df_raw.columns)}")
print(f"Rows    : {len(df_raw):,}")

avg = df_raw.groupby("DGRP")["mn_Longevity"].mean().reset_index()
avg.columns = ["DGRP", PHENO_NAME]
avg["FID"] = "line"
avg["IID"] = avg["DGRP"].str.replace("DGRP_", "", regex=False).str.lstrip("0")
avg = avg[["FID", "IID", PHENO_NAME]].dropna()

fam = pd.read_csv(BASE_DIR / "data" / "reference" / "DGRP.2L.fam",
                  sep=" ", header=None, names=["FID","IID","f","m","s","p"])
overlap = set(avg["IID"].astype(str)) & set(fam["IID"].astype(str))
print(f"Lines averaged : {len(avg)}  |  DGRP2 overlap : {len(overlap)}")

avg.to_csv(PHENO_FILE, sep="\t", index=False)
print(f"Saved : {PHENO_FILE.name}")
avg.head()

##### 7. Genotype QC

Filter DGRP2 SNPs per chromosome arm before GWAS.

| Filter | Value | Reason |
|--------|-------|--------|
| `--maf 0.01` | MAF ≥ 1% | Remove SNPs too rare to test reliably with n = 132 |
| `--geno 0.05` | Missingness ≤ 5% | Remove low-quality genotyping calls |
| `--allow-extra-chr` | — | Required for fly chromosome names (2L, 2R, 3L, 3R, 4, X) |

In [ ]:
qc_dir = gwas_dir / "tmp" / "qc"
qc_dir.mkdir(parents=True, exist_ok=True)
ref_dir = BASE_DIR / "data" / "reference"

print("Per-chromosome QC (MAF >= 0.01, geno <= 0.05):")
for chrom in FLY_CHROMS:
    out = qc_dir / chrom
    if out.with_suffix(".bed").exists():
        n = sum(1 for _ in open(f"{out}.bim"))
        print(f"  {chrom}: {n:,} SNPs (already done)")
        continue
    r = subprocess.run([
        "plink2", "--bfile", str(ref_dir / f"DGRP.{chrom}"),
        "--maf", "0.01", "--geno", "0.05",
        "--allow-extra-chr", "--make-bed", "--out", str(out),
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print(f"  ERROR {chrom}: {r.stderr[-200:]}")
    else:
        n = sum(1 for _ in open(f"{out}.bim"))
        print(f"  {chrom}: {n:,} SNPs after QC")

##### 8. Population Structure PCA

Merge all 6 QC'd chromosome arms genome-wide and compute 10 principal components.
PCs capture genetic ancestry differences among DGRP lines (e.g. cosmopolitan vs.
ancestral strains). PC1 and PC2 will be used as covariates in the GWAS linear model
to regress out population stratification.

In [ ]:
tmp_dir  = gwas_dir / "tmp"
merged   = tmp_dir / "merged_qc"
pca_out  = tmp_dir / "dgrp_pca"
eigenvec = tmp_dir / "dgrp_pca.eigenvec"
eigenval = tmp_dir / "dgrp_pca.eigenval"

if eigenvec.exists():
    print(f"PCA already computed: {eigenvec.name}")
else:
    merge_list = tmp_dir / "pca_merge_list.txt"
    with open(merge_list, "w") as f:
        for ch in FLY_CHROMS[1:]:
            p = qc_dir / ch
            f.write(f"{p}.bed {p}.bim {p}.fam\n")

    print("Merging QC'd chromosomes...")
    r = subprocess.run([
        "plink", "--bfile", str(qc_dir / FLY_CHROMS[0]),
        "--merge-list", str(merge_list), "--allow-extra-chr",
        "--make-bed", "--out", str(merged),
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print("Merge ERROR:", r.stderr[-300:])
    else:
        print("Computing 10 PCs...")
        r2 = subprocess.run([
            "plink2", "--bfile", str(merged), "--allow-extra-chr",
            "--pca", "10", "--out", str(pca_out),
        ], capture_output=True, text=True)
        if r2.returncode != 0:
            print("PCA ERROR:", r2.stderr[-300:])
        else:
            print(f"Done: {eigenvec.name}")

# Load results
pca_df = pd.read_csv(eigenvec, sep="\t")
pca_df.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, len(pca_df.columns)-1)]
print(f"\n{len(pca_df)} lines in eigenvec")
pca_df.head()

##### 8b. PCA Visualization

Two plots:
- **Left**: Scatter of PC1 vs PC2 — each point is one DGRP line. Outlier lines
  (far from the cluster) may indicate divergent ancestry.
- **Right**: Variance explained by each PC (from eigenvalues). In DGRP the
  eigenvalues are roughly uniform, indicating no single dominant axis of structure.

In [ ]:
eigenvec = tmp_dir / "dgrp_pca.eigenvec"
eigenval = tmp_dir / "dgrp_pca.eigenval"

if not eigenvec.exists():
    print("PCA not run yet — execute Section 8 first")
else:
    pca_df = pd.read_csv(eigenvec, sep="\t")
    pca_df.columns = ["FID", "IID"] + [f"PC{i}" for i in range(1, len(pca_df.columns)-1)]

    evals = np.array([float(l.strip()) for l in open(eigenval)])
    var_exp = evals / evals.sum() * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # ── PC1 vs PC2 scatter ──────────────────────────────────────────────────
    ax1.scatter(pca_df["PC1"], pca_df["PC2"], s=30, alpha=0.7, color="#4878CF", edgecolors="white", linewidths=0.3)

    # Label outliers (>2 SD from mean on either axis)
    for ax_col in ["PC1", "PC2"]:
        mu, sd = pca_df[ax_col].mean(), pca_df[ax_col].std()
        outliers = pca_df[np.abs(pca_df[ax_col] - mu) > 2*sd]
        for _, row in outliers.iterrows():
            ax1.annotate(str(row["IID"]), (row["PC1"], row["PC2"]),
                         fontsize=7, ha="left", va="bottom",
                         xytext=(3, 3), textcoords="offset points")

    ax1.set_xlabel(f"PC1 ({var_exp[0]:.1f}% variance explained)")
    ax1.set_ylabel(f"PC2 ({var_exp[1]:.1f}% variance explained)")
    ax1.set_title("DGRP2 Population Structure (n = {} lines)".format(len(pca_df)))
    ax1.axhline(0, color="gray", lw=0.5, ls="--")
    ax1.axvline(0, color="gray", lw=0.5, ls="--")

    # ── Variance explained bar chart ────────────────────────────────────────
    n_pcs = len(evals)
    ax2.bar(range(1, n_pcs+1), var_exp, color="#4878CF", alpha=0.8, edgecolor="white")
    ax2.plot(range(1, n_pcs+1), np.cumsum(var_exp), "o-", color="#D65F5F",
             markersize=4, label="Cumulative")
    ax2.set_xlabel("Principal Component")
    ax2.set_ylabel("Variance Explained (%)")
    ax2.set_title("Scree Plot — Eigenvalue Decomposition")
    ax2.set_xticks(range(1, n_pcs+1))
    ax2.legend()
    ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))

    plt.tight_layout()
    out_png = BASE_DIR / "results" / "dgrp_pca_visualization.png"
    out_png.parent.mkdir(exist_ok=True)
    plt.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png.name}")
    print(f"PC1 variance explained : {var_exp[0]:.2f}%")
    print(f"PC2 variance explained : {var_exp[1]:.2f}%")
    print(f"Top 2 PCs cumulative   : {var_exp[:2].sum():.2f}%")

##### 9. GWAS Association Testing

Run `plink2 --linear` per chromosome arm with **PC1 and PC2 as covariates**.

Including PCs in the model regresses out population stratification so residual
associations reflect true genotype–phenotype effects (same approach as Ivanov et al. 2015
who used the first 4 PCs + Wolbachia status in their DGRP longevity GWAS).

After all chromosomes complete, results are merged, Z-scores computed (Z = BETA / SE),
and sumstats munged with `munge_sumstats.py` for LDSC input.

In [ ]:
SUMSTATS_FILE = gwas_dir / "lifespan_both_sex.sumstats.gz"
MERGED_Z      = gwas_dir / "lifespan_both_sex_z.tsv"

if SUMSTATS_FILE.exists():
    print(f"Sumstats already munged: {SUMSTATS_FILE.name}")
elif not eigenvec.exists():
    print("ERROR: PCA eigenvec not found — run Section 8 first")
else:
    print("Running plink2 linear GWAS with PC1+PC2 covariates...")
    for chrom in FLY_CHROMS:
        out  = tmp_dir / f"lifespan_both_{chrom}"
        glm  = tmp_dir / f"lifespan_both_{chrom}.{PHENO_NAME}.glm.linear"
        if glm.exists():
            print(f"  {chrom}: already done")
            continue
        r = subprocess.run([
            "plink2",
            "--bfile",          str(qc_dir / chrom),
            "--pheno",          str(PHENO_FILE),
            "--pheno-name",     PHENO_NAME,
            "--covar",          str(eigenvec),
            "--covar-col-nums", "3-4",
            "--linear",         "hide-covar",
            "--out",            str(out),
            "--no-psam-pheno",
            "--allow-extra-chr",
        ], capture_output=True, text=True)
        if r.returncode != 0:
            print(f"  ERROR {chrom}: {r.stderr[-200:]}")
        else:
            print(f"  {chrom}: done")

    # Merge chromosomes
    files = sorted(glob.glob(str(tmp_dir / f"lifespan_both_*.{PHENO_NAME}.glm.linear")))
    print(f"\nMerging {len(files)} files...")
    merged_df = pd.concat([pd.read_csv(f, sep="\t") for f in files], ignore_index=True)
    merged_df = merged_df.rename(columns={"#CHROM": "CHR", "ID": "SNP", "OBS_CT": "N"})
    merged_df["Z"] = merged_df["BETA"] / merged_df["SE"]
    bim = pd.concat([
        pd.read_csv(f, sep="\t", header=None, names=["CHR","SNP","CM","BP","A1","A2"])
        for f in sorted(glob.glob(f"{DGRP_PREFIX}.*.bim"))
    ]).drop_duplicates("SNP")
    merged_df = merged_df.merge(bim[["SNP","A2"]], on="SNP", how="left")
    out_df = merged_df[["SNP","A1","A2","Z","N","P"]].dropna()
    out_df.to_csv(MERGED_Z, sep="\t", index=False)
    print(f"Merged: {len(out_df):,} SNPs")

    # Munge
    r = subprocess.run([
        python27_path, str(LDSC_DIR / "munge_sumstats.py"),
        "--sumstats", str(MERGED_Z), "--snp", "SNP", "--a1", "A1", "--a2", "A2",
        "--signed-sumstats", "Z,0", "--N-col", "N",
        "--out", str(gwas_dir / "lifespan_both_sex"),
    ], capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if any(k in line for k in ["SNPs remain","chi^2","Lambda","Written","finished"]):
            print(f"  {line.strip()}")
    if r.returncode == 0:
        print(f"\nLDSC-ready: {SUMSTATS_FILE.name}")

##### 10. GWAS Results: Manhattan & QQ Plots

Visualize association results to confirm inflation is controlled and identify top hits.

- **Lambda GC ≈ 1.0** — well-controlled population stratification
- **Manhattan** — −log10(p) across chromosomes; orange dashed = p = 1×10⁻⁵
- **QQ** — observed vs expected −log10(p); deviation from the diagonal = inflation

In [ ]:
import scipy.stats as st

MERGED_Z = gwas_dir / "lifespan_both_sex_z.tsv"

if not MERGED_Z.exists():
    print("No merged results — run Section 9 first")
else:
    df = pd.read_csv(MERGED_Z, sep="\t")
    files = sorted(glob.glob(str(tmp_dir / f"lifespan_both_*.{PHENO_NAME}.glm.linear")))
    pos = pd.concat([
        pd.read_csv(f, sep="\t", usecols=["#CHROM","ID","POS","P"])
        for f in files], ignore_index=True
    ).rename(columns={"#CHROM":"CHR","ID":"SNP"})
    df = df.merge(pos[["SNP","CHR","POS"]], on="SNP", how="left")
    df["P"] = pd.to_numeric(df["P"], errors="coerce")
    df = df.dropna(subset=["P","POS"]).query("P > 0")

    df["CHR"] = pd.Categorical(df["CHR"].astype(str), categories=FLY_CHROMS, ordered=True)
    df = df.sort_values(["CHR","POS"])
    offset, offsets = 0, {}
    for ch in FLY_CHROMS:
        offsets[ch] = offset
        mx = df[df["CHR"]==ch]["POS"].max()
        if not pd.isna(mx): offset += int(mx) + 1_000_000
    df["cum_pos"] = df.apply(lambda r: r["POS"] + offsets.get(str(r["CHR"]),0), axis=1)
    df["logp"]   = -np.log10(df["P"])

    chi2 = st.chi2.ppf(1 - df["P"].clip(upper=1-1e-15), df=1)
    lam  = float(np.median(chi2) / 0.4549)
    print(f"Lambda GC = {lam:.3f}  ({len(df):,} SNPs)")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    pal = ["#4878CF","#D65F5F"]
    for i, ch in enumerate(FLY_CHROMS):
        s = df[df["CHR"]==ch]
        ax1.scatter(s["cum_pos"], s["logp"], c=pal[i%2], s=1, alpha=0.5, rasterized=True)
    ax1.axhline(-np.log10(1e-5), color="orange", ls="--", lw=0.8, label="p=1e-5")
    ax1.axhline(-np.log10(5e-8), color="red",    ls="--", lw=0.8, label="p=5e-8")
    mids = {ch: df[df["CHR"]==ch]["cum_pos"].median() for ch in FLY_CHROMS}
    ax1.set_xticks([v for v in mids.values() if not pd.isna(v)])
    ax1.set_xticklabels([k for k,v in mids.items() if not pd.isna(v)])
    ax1.set_xlabel("Chromosome"); ax1.set_ylabel("-log10(p)")
    ax1.set_title("Manhattan Plot — Longevity (DGRP2)"); ax1.legend(fontsize=8)

    n = len(df)
    exp = -np.log10(np.arange(1,n+1)/(n+1))
    obs = np.sort(df["logp"].values)[::-1]
    ax2.scatter(exp, obs, s=1, alpha=0.4, color="#4878CF")
    ax2.plot([0,exp.max()],[0,exp.max()],"r--",lw=1)
    ax2.set_xlabel("Expected -log10(p)"); ax2.set_ylabel("Observed -log10(p)")
    ax2.set_title(f"QQ Plot  (λ = {lam:.3f})")

    plt.tight_layout()
    out = BASE_DIR / "results" / "lifespan_gwas_manhattan_qq.png"
    out.parent.mkdir(exist_ok=True)
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out.name}")

##### 11. Build Baseline LD Scores *(optional)*

No pre-built fly baseline exists (unlike human 1000G baseline v1.2).
This cell computes a DGRP-wide baseline LD score file and uses it in h2-cts.
Skip this cell to run h2-cts without a baseline (still valid, slightly less powered).

In [ ]:
baseline_dir = BASE_DIR / "data" / "ldscores" / "baseline"
baseline_dir.mkdir(parents=True, exist_ok=True)

for ch in FLY_CHROMS:
    out = baseline_dir / f"baseline.{ch}.l2.ldscore.gz"
    if out.exists():
        print(f"  chr{ch}: exists")
        continue
    print(f"  Computing baseline chr{ch}...", end=" ", flush=True)
    r = subprocess.run([
        python27_path, str(LDSC_DIR / "ldsc.py"), "--l2",
        "--bfile", f"{DGRP_PREFIX}.{ch}", "--ld-wind-kb", "1000",
        "--out", str(baseline_dir / f"baseline.{ch}"),
    ], capture_output=True)
    print("done" if r.returncode == 0 else f"ERROR: {r.stderr[:100] if r.stderr else ''}")
print("Baseline LD scores ready")

##### 12. Create CTS Reference File

Build the `.cts` tab-delimited file that maps each cell type name to its LD score prefix.
LDSC reads this to know which cell-type LD score files to load for h2-cts.

In [ ]:
results_dir = BASE_DIR / "results"
results_dir.mkdir(exist_ok=True)
CTS_FILE       = BASE_DIR / "data" / "lifespan_both_sex_cell_types.cts"
RESULTS_PREFIX = str(results_dir / "lifespan_both_sex_CellTypeSpecific")

complete_cts = []
for ct in all_cell_types:
    if all((ldscore_dir / ct / f"{ct}.{ch}.l2.ldscore.gz").exists() for ch in FLY_CHROMS):
        complete_cts.append(ct)
    else:
        missing_ch = [c for c in FLY_CHROMS if not (ldscore_dir/ct/f"{ct}.{c}.l2.ldscore.gz").exists()]
        print(f"  {ct}: INCOMPLETE (missing chr {missing_ch})")

with open(CTS_FILE, "w") as f:
    for ct in complete_cts:
        f.write(f"{ct}\t{ldscore_dir}/{ct}/{ct}.\n")

print(f"CTS file: {CTS_FILE.name}  ({len(complete_cts)} cell types)")

##### 13. Run LDSC Cell-Type-Specific Heritability (h2-cts)

Run `ldsc.py --h2-cts` to test whether each cell type's open-chromatin regions
are significantly enriched for longevity heritability.

In [ ]:
SUMSTATS_FILE = gwas_dir / "lifespan_both_sex.sumstats.gz"
baseline_exists = (BASE_DIR / "data" / "ldscores" / "baseline" / "baseline.2L.l2.ldscore.gz").exists()

if not SUMSTATS_FILE.exists():
    print(f"Sumstats not found: {SUMSTATS_FILE.name} — run Section 9 first")
elif not CTS_FILE.exists():
    print(f"CTS file not found — run Section 12 first")
else:
    cmd = [
        python27_path, str(LDSC_DIR / "ldsc.py"),
        "--h2-cts",         str(SUMSTATS_FILE),
        "--ref-ld-chr-cts", str(CTS_FILE),
        "--out",            RESULTS_PREFIX,
    ]
    if baseline_exists:
        cmd += ["--ref-ld-chr", str(BASE_DIR / "data" / "ldscores" / "baseline" / "baseline.")]
        print("Using DGRP baseline LD scores")
    else:
        print("No baseline — running without (Option A)")

    print("Running LDSC h2-cts...")
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2000:] if r.stdout else "")
    if r.returncode != 0:
        print("STDERR:", r.stderr[-500:])
    else:
        print("h2-cts complete")

##### 14. Cell-Type Heritability Enrichment Results

Plot −log10(p) for the top 20 cell types. **Red bars** = p < 0.05.

Each coefficient represents the marginal contribution of that cell type's open-chromatin
regions to SNP heritability, after accounting for all other cell types.

In [ ]:
results_file = RESULTS_PREFIX + ".cell_type_results.txt"

if not os.path.exists(results_file):
    print(f"Results file not found: {results_file}")
    print("Run Section 13 first.")
else:
    res = pd.read_csv(results_file, sep="\t")
    res = res.sort_values("Coefficient_P_value").reset_index(drop=True)
    res["logp"] = -np.log10(res["Coefficient_P_value"])
    res["sig"]  = res["Coefficient_P_value"] < 0.05

    top = res.head(20).iloc[::-1]

    fig, ax = plt.subplots(figsize=(9, max(4, len(top) * 0.38)))
    colors = ["#C0392B" if s else "#7FB3D3" for s in top["sig"]]
    ax.barh(top["Name"], top["logp"], color=colors, height=0.7)
    ax.axvline(-np.log10(0.05), color="black", ls="--", lw=0.8, label="p = 0.05")
    ax.set_xlabel("-log10(p-value)")
    ax.set_title("Cell-Type Heritability Enrichment — Longevity (DGRP2)")
    ax.legend(fontsize=8)
    plt.tight_layout()
    out = BASE_DIR / "results" / "lifespan_cts_enrichment.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out.name}")

    print(f"\nSignificant cell types (p < 0.05) : {res['sig'].sum()}")
    sig = res[res["sig"]][["Name","Coefficient","Coefficient_std_error","Coefficient_P_value"]]
    print(sig.to_string(index=False))
    print(f"\nTop 10 by p-value:")
    print(res[["Name","Coefficient","Coefficient_P_value"]].head(10).to_string(index=False))